# Lecture 08 — Anti-bot, and when not to scrape

> *"The best way to win the cat-and-mouse game is to not be in it."*

In the previous seven lectures we acted as if the web were neutral terrain. It isn't. Every commercially valuable site sits behind some combination of rate-limiting, fingerprinting, behavioural analysis, and outright blocking. The arms race between scrapers and defenders is real, expensive, and — if you're a one-person tutorial student — almost always one you should refuse to enter.

This lecture has two halves. The first describes the landscape so you can recognise what you're up against. The second is a sober look at when the right move is to put the keyboard down: ask, license, partner, or walk away.

## What you'll be able to do after this lecture

- Recognise the major anti-bot mechanisms (Cloudflare, DataDome, PerimeterX, captchas, fingerprinting) when you encounter them.
- Understand what each defence actually checks (IP, headers, TLS, JS, behaviour).
- Reason about countermeasures — proxies, residential IPs, stealth browsers, captcha solvers — and *their* costs.
- Apply a decision tree that includes "don't" and "ask the owner" as first-class options.


## 1. The defender's toolbox

A modern anti-bot stack layers cheap checks under expensive ones. Each layer rejects bots that don't bother to clear it, so the next layer only sees the more determined traffic.

| Layer | What it checks | How to recognise it as a scraper |
|-------|----------------|-----------------------------------|
| Rate limit | Requests per IP per minute | HTTP `429`, `Retry-After` header |
| User-Agent / headers | Presence + plausibility of browser headers | `403` or empty page on `python-httpx/...` UA |
| IP reputation | Datacentre vs residential, blocklists | `403` even with perfect headers; works from your laptop, fails from AWS |
| TLS fingerprint (JA3/JA4) | Cipher order, ALPN, extensions | Block fires *before* HTTP — connection-level reject |
| HTTP/2 fingerprint | Frame ordering, settings | Same as above |
| JavaScript challenge | Can the client run JS? | An interstitial page that runs `eval()`-y code, then redirects |
| Cookie / token issuance | Did the JS challenge mint a cookie? | Subsequent requests need a `cf_clearance`-style token |
| Behavioural | Mouse movement, scroll cadence, focus events | Bot detected even after rendering JS — telemetry from `mouseover`/`pointermove` |
| CAPTCHA | Human-challenge fallback | reCAPTCHA, hCaptcha, Cloudflare Turnstile widget appears |

The higher you go, the harder and more expensive each defence is to bypass — and the clearer the signal that the site genuinely doesn't want you scraping.


## 2. Fingerprinting, in slightly more detail

A "fingerprint" is any combination of signals the server can use to tell *this client* apart from *that client*. The more layers you forget about, the more you stand out.

- **IP address.** Datacentre ranges (AWS, GCP, Hetzner) are flagged by reputation services. Residential IPs look human; that's why "residential proxy" services exist and cost real money.
- **HTTP headers.** Real browsers send a specific *order* of `User-Agent`, `Accept`, `Accept-Language`, `Accept-Encoding`, `sec-ch-ua-*`, `sec-fetch-*`. `httpx` won't send half of these. Using only `User-Agent: Mozilla/5.0 ...` while sending no `Accept-Language` is a giveaway.
- **TLS fingerprint (JA3 / JA4).** Your TLS Client Hello — cipher suites, supported groups, ALPN — has a recognisable shape. Python's TLS stack produces a different shape than Chrome's. Tools exist to spoof it (`curl-impersonate`, `httpx[brotli]+ja3-spoofing`); none of them are turnkey.
- **HTTP/2 frame fingerprint.** The order in which a client sends `SETTINGS` and `HEADERS` frames also identifies the library. Defences as sophisticated as Akamai's check this.
- **JavaScript runtime.** Headless Chromium sets `navigator.webdriver = true` by default. It also leaks via `navigator.plugins.length`, `chrome.runtime`, missing `WebGL` extensions, mismatched timezone, missing fonts. Stealth plugins try to patch the obvious ones; the long tail is endless.
- **Behaviour.** Real users move the mouse before clicking, scroll erratically, take 200–800ms between actions, and pause to read. Bots don't, by default.

Notice the trend: each layer is harder to fake than the last, and at some point you're effectively building a browser. That's the trap.


## 3. The big anti-bot vendors

If you scrape commercially relevant sites, you'll meet these by name:

- **Cloudflare** — fronts a huge fraction of the web. "Just a moment..." interstitials, `cf_clearance` cookies, Turnstile captchas. Bot Fight Mode and Bot Management are paid tiers most large sites use.
- **DataDome** — heavy JS challenges, behavioural telemetry, often the silent block on retail and ticketing sites.
- **PerimeterX / HUMAN** — similar, popular on travel and e-commerce.
- **Akamai Bot Manager** — common on banking, airlines. TLS- and HTTP/2-level fingerprinting.
- **Imperva (Incapsula)** — older but still around.

How to tell which is in front of you:

- Look at response headers (`Server`, `cf-ray`, `x-iinfo`).
- Look at cookies the page sets (`cf_clearance`, `datadome`, `_pxhd`, `incap_ses_*`).
- Look at the HTML of the block page — they're each branded.

Acknowledging *which* defence you've hit also tells you *roughly how serious the operator is*. A site behind enterprise Akamai is paying real money to keep you out. Read that signal.


## 4. CAPTCHAs

Captchas are the explicit human-check fallback. The common ones in 2026:

- **reCAPTCHA v2** — "I'm not a robot" checkbox plus image grids. Mostly a scoring system; the checkbox passes if your prior signals are clean.
- **reCAPTCHA v3** — invisible scoring, no widget. The site decides what to do with your score.
- **hCaptcha** — privacy-friendlier alternative; image grids, used by Cloudflare for years before they switched to Turnstile.
- **Cloudflare Turnstile** — "managed challenge" that may or may not show UI; designed for low friction.
- **Custom puzzle / slider** — Chinese e-commerce favours these. Often the easiest to solve programmatically, hardest to do *politely*.

Third-party "captcha solving" services (2Captcha, Anti-Captcha, CapSolver) outsource the puzzles to humans or ML for cents per solve. They work, they're cheap, and using them at scale on a site that explicitly deployed captchas to keep bots out is — at minimum — bad faith. In some jurisdictions and contexts it's also a CFAA / unauthorised-access problem. Tread carefully.


## 5. Counter-techniques (and their real costs)

It is technically possible to defeat most defences. Here's what each costs you, in money and in moral position.

| Counter-technique | Money | Engineering | Moral position |
|-------------------|-------|-------------|----------------|
| Realistic headers (`User-Agent`, `Accept-Language`, etc.) | Free | Low | Fine — basic politeness. |
| Slowing down (1 req/sec) | Free | Low | Fine — exactly what `Crawl-delay` asks for. |
| Datacentre proxies | Cheap (~$1/GB) | Low | Mostly fine. |
| Residential proxies | Expensive (~$5–15/GB) | Medium | Often involves "residential" IPs harvested from people who installed dubious VPN apps. Read who you're paying. |
| Headful Playwright with stealth plugins | Free + compute | Medium | Fine for one-off; arms-race-y at scale. |
| TLS impersonation (`curl-impersonate`) | Free | Medium | Crossing into "actively pretending to be a different program." |
| Captcha-solving services | Cheap per solve | Low | The site deployed captchas *to stop bots*. You are a bot. |
| Buying scraped data from a vendor | Varies | None | Cleaner if the vendor is legitimate (they often aren't). |
| Talking to the site owner | Free | Lots of social effort | Best by a wide margin. |

Notice the entries that are free and low-engineering are also the ones with the cleanest moral position. That isn't a coincidence — they're the ones that stay inside the social contract of the web.


In [ ]:
# Politeness-first request: identify yourself, slow down, respect 429.
import httpx
import time

HEADERS = {
    "User-Agent": "CrawlingTutorial/0.1 (+https://example.com/contact; vova.e.125@gmail.com)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}


def polite_get(url: str, *, max_retries: int = 3) -> httpx.Response:
    with httpx.Client(headers=HEADERS, timeout=15, http2=True) as c:
        for attempt in range(max_retries):
            r = c.get(url)
            if r.status_code == 429:
                wait = int(r.headers.get("Retry-After", 2 ** attempt))
                print(f"  429; sleeping {wait}s")
                time.sleep(wait)
                continue
            return r
        raise RuntimeError("too many 429s; back off and try later")


r = polite_get("https://httpbin.org/headers")
print(r.status_code)
print(r.json())


Identifying yourself in your `User-Agent` (with a contact URL or email) is small, costs nothing, and turns you from "anonymous bot" into "a person we could email." When operators dislike scraping, they generally dislike *unaccountable* scraping; being reachable solves much of that.


## 6. The underrated option: don't

A half-finished, half-blocked scraper that produces stale data once a week is worse than nothing. Before you spend a weekend bypassing Cloudflare, check whether you actually have to.

- **Is there a paid API?** $20/month for a clean feed beats $200/month in residential proxy bandwidth and a constant maintenance burden.
- **Is there a data partner?** Bloomberg, Crunchbase, SimilarWeb, etc. exist because someone decided not to scrape.
- **Can you ask?** A polite email — "I'm a researcher / student / building X, I need this data Y times a week, can we work something out?" — succeeds far more often than scrapers expect. Operators sometimes give you a CSV. Sometimes they invite you to the API beta. Worst case, they say no and you're back where you started.
- **Is the scope smaller than you think?** Maybe you only need 200 records, not 200k. Maybe a one-time manual export is fine. *Don't crawl the encyclopaedia when you need a paragraph.*
- **Are you actually the right party to do this?** A research dataset that the BBC can publish in 30 minutes shouldn't take you 3 weeks of cat-and-mouse to scrape.

The rule of thumb: **if the site is fighting you, the site disagrees with what you're doing.** Sometimes they're wrong (price-comparison aggregators have argued this in court, with mixed results). Often they're right. Default to assuming it's the latter.


## 7. Legal & ethical reframe

This is not a legal lecture and your jurisdiction matters, but a few stable points:

- **Public data isn't permission.** Something being viewable doesn't grant a licence to bulk-collect it. Copyright, ToS, and computer-misuse statutes can all apply.
- **Bypassing technical access controls** (logins, captchas, IP blocks) lands closer to *unauthorised access* than "ToS violation" in most legal systems. The hiQ v. LinkedIn line of cases is narrow and US-specific.
- **Personal data** (anything that can identify a person) is regulated by GDPR (EU), CCPA (California), PIPEDA (Canada), PIPA (Korea), etc. Scraping into a database is *processing*, with all that follows.
- **Robots.txt is informational, not legal**, but ignoring `Disallow:` is a poor showing in any dispute.
- **Don't redistribute scraped data without thinking.** Aggregating someone else's content into your own product is a different conversation than analysing it for your own research.

If you're scraping at any scale that involves money — yours or someone else's — get a lawyer. The cost is far less than the cost of being wrong.


## 8. Decision tree

```
Do I need this data?
│
├── Can I get it from a paid API or partner?    ─yes─> Pay them. Done.
├── Can I ask the operator nicely?              ─yes─> Email; wait; often works.
├── Is the site happy to be crawled?            ─yes─> Identify yourself, go slow, lecture 06.
├── Is the site mildly resistant (rate limits)? ─yes─> Slow down further. Cache. Re-evaluate.
├── Is the site behind serious anti-bot?        ───
│        │
│        ├── Is this for personal learning, low volume? ─> Maybe — but consider this lecture's framing.
│        ├── Is this for production / commercial? ─────> Stop. Buy the data or partner.
│        └── Are you sure you're not the bad actor here? ─> Sit with that for a minute.
└── Is this somebody's personal data?           ─yes─> Privacy law. Different course.
```

Notice that more than half the branches resolve *without* writing more code. That's intentional — most scraping decisions are organisational, not technical.


## 9. If you must: the minimum-effort dignified scrape

When you do decide it's appropriate to scrape a mildly defended site, in roughly this order:

1. Identify yourself in `User-Agent`, with a way to contact you.
2. Respect `robots.txt` and `Crawl-delay`.
3. Cache aggressively — never refetch a page in the same week unless you have to.
4. Limit concurrency (lecture 06).
5. Backoff on `429` and `5xx`.
6. Use `httpx` with a realistic browser-like header set, not a stealth-browser farm.
7. Stop and reconsider the moment you see Cloudflare's interstitial, a captcha, or a behavioural challenge.

If step 7 fires, the site has actively told you no. Treat that as a no.


## Recap

- Modern anti-bot is a layered stack: rate limits, header checks, IP reputation, TLS/HTTP fingerprinting, JS challenges, behavioural telemetry, captchas.
- Each layer you bypass is more engineering, more cost, and a worse moral position.
- Cloudflare, DataDome, PerimeterX, Akamai are the names you'll meet most often. Learn to recognise their fingerprints in headers and cookies.
- The cheapest, most underrated technique is *not scraping*: ask, license, partner, narrow the scope, or walk away.
- If you must scrape: identify yourself, go slow, cache, and stop the moment the site says no.

## Exercises

1. Pick three sites you'd actually like to scrape. For each, open DevTools and identify which anti-bot stack is in front (look at headers and cookies). Write down what you see.
2. Find a site whose `robots.txt` *invites* crawling (Wikipedia, GitHub, gov.uk are good candidates). Write a 2-paragraph contrast with a site that clearly doesn't want it.
3. For a site you'd want data from, draft the email you'd send the operator. One paragraph: who you are, what you want, why, how often. Don't send it; just write it.
4. Look up the hiQ v. LinkedIn case (US, 2017–2022). Summarise in 200 words why this lecture's framing is *not* "scraping is illegal," but also not "go nuts."

## Up next

**Lecture 09** — when selectors are the wrong tool entirely: using LLMs to extract structured data from messy pages. Where it shines, where it doesn't, and what it costs.
